In [ ]:
import os
import re
import pandas as pd
import numpy as np
import nltk
import spacy
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Download necessary NLP data packages
nltk.download('stopwords')
nltk.download('punkt')

# Load spaCy English core model
try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    import os
    os.system("python -m spacy download en_core_web_sm")
    nlp = spacy.load("en_core_web_sm")

print("All libraries imported and NLP models loaded successfully!")

In [ ]:
# Create a data directory if it doesn't exist
os.makedirs("data", exist_ok=True)

try:
    # Attempt to load your downloaded Kaggle CSVs
    resumes_df = pd.read_csv("data/Resume.csv")
    jobs_df = pd.read_csv("data/monster_com-job_sample.csv")
    
    # Standardize column naming for compatibility
    if 'Resume_str' in resumes_df.columns:
        resumes_df['resume_text'] = resumes_df['Resume_str']
    elif 'Resume' in resumes_df.columns:
        resumes_df['resume_text'] = resumes_df['Resume']
        
    if 'job_description' in jobs_df.columns:
        jobs_df['jd_text'] = jobs_df['job_description']
        
    print(f"Successfully loaded {len(resumes_df)} resumes and {len(jobs_df)} jobs from CSV files!")

except FileNotFoundError:
    print("⚠️ CSV files not found in 'data/' directory. Creating a quick mock dataset for validation...")
    
    # Mock Resumes
    resume_data = {
        'ID': [101, 102, 103],
        'Category': ['Data Science', 'Web Developer', 'HR Manager'],
        'resume_text': [
            "John Doe. Data Scientist with 3 years experience. Skilled in Python, SQL, Machine Learning, and AWS cloud deployment. Built predictive algorithms.",
            "Jane Smith. Frontend Web Developer. Expert in JavaScript, HTML, CSS, Git, and React framework. Passionate about UI/UX design.",
            "Alice Johnson. HR Specialist. Experienced in talent acquisition, recruitment, employee onboarding, and project management using Agile frameworks."
        ]
    }
    resumes_df = pd.DataFrame(resume_data)
    
    # Mock Job Descriptions
    job_data = {
        'job_title': ['Data Scientist', 'Frontend Engineer'],
        'jd_text': [
            "We are looking for a Data Scientist proficient in Python, SQL, Machine Learning, and Docker. Experience with AWS is a major plus.",
            "Hiring a Web Developer with strong skills in Javascript, React, HTML, and CSS. Knowledge of Git is required."
        ]
    }
    jobs_df = pd.DataFrame(job_data)
    print("Mock datasets initialized successfully!")

In [ ]:
stop_words = set(stopwords.words('english'))

def clean_text(text):
    if not isinstance(text, str):
        return ""
    
    # 1. Convert to lowercase
    text = text.lower()
    # 2. Remove URLs
    text = re.sub(r'http\S+\s*', ' ', text)
    # 3. Remove email addresses
    text = re.sub(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b', ' ', text)
    # 4. Remove special characters, punctuation, and numbers
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    # 5. Remove extra whitespace collapses
    text = re.sub(r'\s+', ' ', text).strip()
    
    # 6. Remove non-informative stopwords
    text_tokens = text.split()
    filtered_words = [word for word in text_tokens if word not in stop_words]
    
    return " ".join(filtered_words)

# Apply cleaning script to datasets
resumes_df['cleaned_resume'] = resumes_df['resume_text'].apply(clean_text)
jobs_df['cleaned_jd'] = jobs_df['jd_text'].apply(clean_text)

print("Sample cleaned resume text:\n", resumes_df['cleaned_resume'].iloc[0][:150] + "...")

In [ ]:
from spacy.matcher import PhraseMatcher

# Comprehensive list of standard skills to scan for
SKILL_DATABASE = [
    'python', 'sql', 'java', 'c++', 'javascript', 'tableau', 'power bi', 
    'excel', 'machine learning', 'deep learning', 'aws', 'azure', 'git', 
    'docker', 'kubernetes', 'project management', 'agile', 'scrum', 
    'html', 'css', 'react', 'recruitment', 'onboarding'
]

def extract_skills(text):
    matcher = PhraseMatcher(nlp.vocab, attr="LOWER")
    patterns = [nlp.make_doc(skill) for skill in SKILL_DATABASE]
    matcher.add("SkillPatterns", patterns)
    
    doc = nlp(text)
    matches = matcher(doc)
    
    extracted = set()
    for match_id, start, end in matches:
        extracted.add(doc[start:end].text.lower())
    return list(extracted)

# Map found skills to respective dataframes
resumes_df['extracted_skills'] = resumes_df['cleaned_resume'].apply(extract_skills)
jobs_df['extracted_skills'] = jobs_df['cleaned_jd'].apply(extract_skills)

print("Extraction test complete. Identified skills in Resume 0:", resumes_df['extracted_skills'].iloc[0])

In [ ]:
def screen_candidates(target_job_index=0):
    # Select the specific job description to test against
    target_job = jobs_df.iloc[target_job_index]
    target_jd_clean = target_job['cleaned_jd']
    target_jd_skills = set(target_job['extracted_skills'])
    job_title = target_job.get('job_title', f"Job Profile #{target_job_index}")
    
    print(f"🎯 Screening applicants for open role: '{job_title}'")
    print(f"📋 Required skills found in Job Description: {list(target_jd_skills)}\n")
    
    scores = []
    matched_skills_list = []
    missing_skills_list = []
    
    # Process each resume entry against the selected job
    for idx, row in resumes_df.iterrows():
        # 1. Compute text similarity score using TF-IDF
        vectorizer = TfidfVectorizer()
        tfidf_matrix = vectorizer.fit_transform([row['cleaned_resume'], target_jd_clean])
        similarity = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:2])[0][0]
        match_percentage = round(similarity * 100, 2)
        scores.append(match_percentage)
        
        # 2. Map intersection and structural skill gaps
        candidate_skills = set(row['extracted_skills'])
        matched = target_jd_skills.intersection(candidate_skills)
        missing = target_jd_skills - candidate_skills
        
        matched_skills_list.append(list(matched))
        missing_skills_list.append(list(missing))
        
    # Append calculated evaluation metrics back to a clean copy of the dataframe
    results_df = resumes_df.copy()
    results_df['match_score (%)'] = scores
    results_df['matched_skills'] = matched_skills_list
    results_df['missing_skills'] = missing_skills_list
    
    # Sort data by highest match percentage score
    ranked_df = results_df.sort_values(by='match_score (%)', ascending=False)
    
    return ranked_df

# Run execution framework on the initial job entry
final_rankings = screen_candidates(target_job_index=0)

# Display final report structure output
display_cols = ['Category', 'match_score (%)', 'extracted_skills', 'missing_skills']
if 'ID' in final_rankings.columns:
    display_cols.insert(0, 'ID')

print("====== FINAL CANDIDATE RANKING TABLE ======")
final_rankings[display_cols].head()